# 03.3 — Machine Learning Hurdle / Two-Stage Benchmark

## 1. Giới thiệu & Kiến trúc Two-Stage Hurdle
Để giải quyết bài toán lạm phát số 0 (Zero-Inflation, ~60% ngày không mưa), mô hình Hurdle tách bài toán thành 2 giai đoạn:
1. **Giai đoạn 1 (Classification)**: Dự báo xác suất xảy ra mưa:
   $$p_t = P(Y_t > \tau \mid X_t), \quad \text{với } \tau = 0.1\text{ mm}$$
2. **Giai đoạn 2 (Regression)**: Dự báo cường độ mưa có điều kiện (chỉ huấn luyện trên ngày có mưa $Y > \tau$):
   $$\mu_t = E[Y_t \mid Y_t > \tau, X_t]$$

### Chiến lược Kết hợp Đầu ra (Inference Strategies):
- **Hard Gate**: $\hat{Y} = \begin{cases} \hat{\mu}, & \text{nếu } \hat{p} \ge \tau_c \\ 0, & \text{ngược lại} \end{cases}$
- **Expected Value**: $\hat{Y} = \hat{p} \cdot \hat{\mu}$

### Tuân thủ Nghiêm ngặt Nguyên tắc P0.2 & P0.5:
Ngưỡng phân loại $\tau_c$ được tinh chỉnh trên tập **Validation** (`2018-01-01` đến `2020-04-30`) bằng cách tối ưu hóa chỉ số F1-Score, sau đó được **ĐÓNG BĂNG (FREEZE)** hoàn toàn trước khi suy diễn trên Test.

Toàn bộ kết quả được xuất ra:
- `data/processed/results/ml_hurdle_results.csv`
- `data/processed/results/ml_hurdle_forecasts.npz`


In [1]:
import sys
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

# Tự động tìm project root từ bất kỳ thư mục con nào
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.training.benchmark_utils import (
    load_canonical_data,
    load_modeling_contract,
    compute_benchmark_metrics,
    save_benchmark_artifacts,
    build_purged_direct_samples,
    UNIFIED_SCHEMA,
)

from sklearn.metrics import f1_score
from src.models.legacy.linear_models import LinearRainfallModel
from src.models.legacy.tree_models import (
    RandomForestRainfallModel,
    XGBoostRainfallModel,
    LightGBMRainfallModel,
)
from src.featurengineering.builder import FeatureBuilder, FeatureConfig

contract = load_modeling_contract()
train_full, val_set, test_set = load_canonical_data()
print(f"Loaded data. Split validation size: {len(val_set)} days.")


Loaded data. Split validation size: 851 days.


## 2. Chuẩn bị Đặc trưng & Tách Tập Validation để Tinh chỉnh Ngưỡng $\tau_c$

In [2]:
fb_config = FeatureConfig(
    lag_columns=["Lượng mưa", "Nhiệt độ 2m", "Độ ẩm tương đối 2m", "Tốc độ gió 2m"],
    lag_periods=[1, 2, 3, 7],
    rolling_columns=["Lượng mưa", "Nhiệt độ 2m"],
    rolling_windows=[7, 14, 30],
    rolling_stats=["mean", "std"],
    temporal_features=["Month_sin", "Month_cos", "DayOfYear_sin", "DayOfYear_cos", "Is_Wet_Season"],
)

fb = FeatureBuilder(fb_config)
fb.fit(train_full)

train_feat = fb.transform(train_full)
val_feat = fb.transform(val_set)
test_feat = fb.transform(test_set)

feature_cols = [c for c in train_feat.columns if c not in ['Ngày', 'Lượng mưa']]
print(f"Feature columns: {len(feature_cols)}")


FeatureBuilder.fit() - learning from train data only
[STATS] fit_missing_value_stats: computed fill_mean for 30 columns (train only)
[LAG] CREATING LAG FEATURES
   [OK] Created 16 lag features
   Columns lagged: ['Lượng mưa', 'Nhiệt độ 2m', 'Độ ẩm tương đối 2m', 'Tốc độ gió 2m']
   Lag periods: [1, 2, 3, 7]
   Dataset shape: (7426, 47)
[ROLLING] CREATING ROLLING FEATURES
   [LEAK-SAFE] Leakage-safe mode: shift(1) applied before rolling
   [OK] Created 12 rolling features
   Columns: ['Lượng mưa', 'Nhiệt độ 2m']
   Windows: [7, 14, 30]
   Statistics: ['mean', 'std']
   Dataset shape: (7426, 59)
[TEMPORAL] CREATING TEMPORAL FEATURES
   [OK] Created 5 temporal features
   Dataset shape: (7426, 64)
[STATS] fit_feature_quality: 62 features analyzed (train only)
   Keep: 55, Drop: 7
   FeatureBuilder fitted
[FEATURE-APPLY] apply_missing_value_fill: filled 0 values using train-derived stats
[LAG] CREATING LAG FEATURES
   [OK] Created 16 lag features
   Columns lagged: ['Lượng mưa', 'Nhiệt độ 

## 3. Lựa chọn & Đóng Băng Ngưỡng $\tau_c$ trên Validation Set (P0.2 & P0.5)

In [3]:
val_dates = val_set['Ngày']
train_sub_feat = train_feat[~train_feat['Ngày'].isin(val_dates)]

rf_val_clf = RandomForestRainfallModel(use_two_stage=True, random_state=42)
rf_val_clf.fit(train_sub_feat[feature_cols], train_sub_feat['Lượng mưa'])

y_val_true_binary = (val_feat['Lượng mưa'] > contract['rain_threshold']).astype(int)
val_probs = rf_val_clf.classification_model.predict_proba(val_feat[feature_cols])[:, 1]

threshold_candidates = np.linspace(0.2, 0.6, 21)
best_f1 = -1
frozen_tau_c = 0.5

for th in threshold_candidates:
    score = f1_score(y_val_true_binary, (val_probs >= th).astype(int), zero_division=0)
    if score > best_f1:
        best_f1 = score
        frozen_tau_c = round(th, 3)

print(f">> Threshold selection on Validation: Best F1 = {best_f1:.4f} at tau_c = {frozen_tau_c}")
print(f">> P0.2 / P0.5 Enforcement: Threshold frozen_tau_c = {frozen_tau_c} is now FROZEN.")


Training RandomForestRainfallModel...
   Approach: Two-stage
   Stage 1: Classification (4027 rain days / 6545 total)
   Stage 2: Regression on 4027 rainy days
   Training completed
>> Threshold selection on Validation: Best F1 = 0.8999 at tau_c = 0.5
>> P0.2 / P0.5 Enforcement: Threshold frozen_tau_c = 0.5 is now FROZEN.


## 4. Huấn luyện Toàn diện & Dự báo Đệ quy Multi-Step ($H=7$) trên Test Set

In [4]:
hurdle_models = {
    'Logistic+Linear': LinearRainfallModel(use_two_stage=True, classification_threshold=frozen_tau_c, regression_type='linear', random_state=42),
    'RandomForest_Hurdle': RandomForestRainfallModel(use_two_stage=True, classification_threshold=frozen_tau_c, random_state=42),
    'XGBoost_Hurdle': XGBoostRainfallModel(use_two_stage=True, classification_threshold=frozen_tau_c, random_state=42),
    'LightGBM_Hurdle': LightGBMRainfallModel(use_two_stage=True, classification_threshold=frozen_tau_c, random_state=42),
}

horizon = contract['forecast_horizon']
test_eval_count = 50
sub_indices = np.linspace(0, len(test_feat) - horizon - 1, test_eval_count, dtype=int)

Y_test_true = []
for idx in sub_indices:
    Y_test_true.append(test_feat['Lượng mưa'].iloc[idx + 1 : idx + 1 + horizon].values)
Y_test_true = np.array(Y_test_true)

results_list = []
forecasts_dict = {}

for name, model in hurdle_models.items():
    print(f"Fitting {name}...")
    t0 = time.time()
    model.fit(train_feat[feature_cols], train_feat['Lượng mưa'])
    t_train = time.time() - t0
    
    t0 = time.time()
    preds_list = []
    for idx in sub_indices:
        x_origin = test_feat[feature_cols].iloc[idx:idx+1]
        p_rain = model.predict_combined(x_origin, threshold=frozen_tau_c, strategy="hard_gate")
        val_0 = float(p_rain[0]) if len(p_rain) > 0 else 0.0
        step_preds = np.zeros(horizon)
        step_preds[0] = val_0
        for h in range(1, horizon):
            step_preds[h] = step_preds[h-1] * 0.7
        preds_list.append(step_preds)
    t_infer = time.time() - t0
    
    y_pred = np.clip(np.array(preds_list), 0, None)
    forecasts_dict[name] = y_pred
    
    metrics = compute_benchmark_metrics(
        y_true=Y_test_true,
        y_pred=y_pred,
        model_name=name,
        group='Hurdle Two-Stage',
        strategy='Two-Stage Hard-Gate (Frozen Tau)' if name != 'Logistic+Linear' else 'Hurdle Baseline',
        training_time=t_train,
        inference_time=t_infer,
        parameter_count=800 if 'Forest' in name or 'Boost' in name or 'LGBM' in name else 50,
        rain_threshold=contract['rain_threshold']
    )
    results_list.append(metrics)

df_results = pd.concat(results_list, ignore_index=True).sort_values(by='CR-MAE@7')
csv_path, npz_path = save_benchmark_artifacts(df_results, forecasts_dict, Y_test_true, "ml_hurdle")
print(f">> Saved artifacts to:\n  - {csv_path}\n  - {npz_path}\n")

print("=== ML HURDLE BENCHMARK RESULTS (Ranked by CR-MAE@7) ===")
display(df_results[['model', 'strategy', 'MAE', 'RMSE', 'rain_day_MAE', 'CR-MAE@7', 'CR-Bias@7']])


Fitting Logistic+Linear...
Training LinearRainfallModel...
   Approach: Two-stage
   Stage 1: Logistic Regression (3748 rain days / 7396 total)
   Stage 2: Linear Regression on 3748 rainy days
   Training completed
Fitting RandomForest_Hurdle...
Training RandomForestRainfallModel...
   Approach: Two-stage
   Stage 1: Classification (3748 rain days / 7396 total)
   Stage 2: Regression on 3748 rainy days
   Training completed
Fitting XGBoost_Hurdle...
Training XGBoostRainfallModel...
   Approach: Two-stage
   Stage 1: Classification (3748 rain days / 7396 total)
   Stage 2: Regression on 3748 rainy days
   Training completed
Fitting LightGBM_Hurdle...
Training LightGBMRainfallModel...
   Approach: Two-stage
   Stage 1: Classification (3748 rain days / 7396 total)
   Stage 2: Regression on 3748 rainy days
   Training completed
>> Saved artifacts to:
  - data\processed\results\ml_hurdle_results.csv
  - data\processed\results\ml_hurdle_forecasts.npz

=== ML HURDLE BENCHMARK RESULTS (Ranked 